In [1]:

# %%
# ==============================================
# Week 4 Section 3 — Model Selection & Trade-offs (Business)
# Reads Section 2 metrics and summarizes champion models
# ==============================================

PROFILE = "dev"   # dev | preprod | final
SECTION_ID = "Wk04_Section3"

from pathlib import Path
BASE_DIR = Path.cwd()
if BASE_DIR.name.lower() == "code":
    BASE_DIR = BASE_DIR.parent
RESULTS_DIR = BASE_DIR / "results" / f"{SECTION_ID}_{PROFILE}"
REPORTS_DIR = RESULTS_DIR / "reports"
PLOTS_DIR   = RESULTS_DIR / "plots"
for d in (REPORTS_DIR, PLOTS_DIR): d.mkdir(parents=True, exist_ok=True)

CONSOLIDATED = BASE_DIR / "SDS-CP036-powercast_Wk04_Report_Business.md"
S2_METRICS = BASE_DIR / "results" / f"Wk04_Section2_{PROFILE}" / "reports" / "retrain_test_metrics.csv"


# Common helpers
from pathlib import Path
import os, re, math, warnings, json
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

# Cap threads to keep runtime light
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["VECLIB_MAXIMUM_THREADS"] = "1"
os.environ["NUMEXPR_NUM_THREADS"] = "1"
os.environ["MALLOC_ARENA_MAX"] = "2"
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

# Optional libraries
_HAVE_SM = False
try:
    from statsmodels.tsa.statespace.sarimax import SARIMAX
    _HAVE_SM = True
except Exception:
    _HAVE_SM = False

_HAVE_FBPROPHET = False
try:
    try:
        from prophet import Prophet
        _HAVE_FBPROPHET = True
    except Exception:
        from fbprophet import Prophet
        _HAVE_FBPROPHET = True
except Exception:
    _HAVE_FBPROPHET = False

_HAVE_XGB = False
try:
    from xgboost import XGBRegressor as _XGB
    def XGBRegressor(*args, **kwargs):
        kwargs.setdefault("n_estimators", 200)
        kwargs.setdefault("max_depth", 4)
        kwargs.setdefault("learning_rate", 0.1)
        kwargs.setdefault("subsample", 0.9)
        kwargs.setdefault("colsample_bytree", 0.9)
        kwargs.setdefault("tree_method", "hist")
        kwargs.setdefault("n_jobs", 1)
        return _XGB(*args, **kwargs)
    _HAVE_XGB = True
except Exception:
    _HAVE_XGB = False

def normalize_columns(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df.columns = [re.sub(r"\s+", " ", c.strip()) for c in df.columns]
    return df

def infer_step_and_freq(idx: pd.DatetimeIndex):
    try:
        freq = pd.infer_freq(idx[:1000])
    except Exception:
        freq = None
    if freq:
        step = pd.to_timedelta(pd.tseries.frequencies.to_offset(freq))
        return step, freq
    diffs = pd.Series(idx).diff().dropna()
    step = diffs.mode().iloc[0] if not diffs.empty else pd.Timedelta(hours=1)
    total_sec = int(step.total_seconds())
    if total_sec % 3600 == 0:
        freq = f"{total_sec // 3600}H"
    elif total_sec % 60 == 0:
        freq = f"{total_sec // 60}T"
    else:
        freq = f"{total_sec}S"
    return step, freq

def limit_last_days(df: pd.DataFrame, date_col="DateTime", max_days=None) -> pd.DataFrame:
    if not max_days:
        return df
    df = df.copy()
    df[date_col] = pd.to_datetime(df[date_col])
    df = df.sort_values(date_col)
    cutoff = df[date_col].max() - pd.Timedelta(days=max_days)
    return df[df[date_col] >= cutoff]

def pick_zone_cols(df: pd.DataFrame, k: int):
    cols = [c for c in df.columns if re.match(r"Zone\s+\d+\s+Power\s+Consumption", c)]
    cols = sorted(cols, key=lambda s: int(re.search(r"\d+", s).group()))
    return cols[:k] if k else cols

def make_time_splits(df: pd.DataFrame, test_days: int, date_col="DateTime", steps_per_day: int = 24):
    df = df.sort_values(date_col)
    test_len = int(test_days * steps_per_day)
    if test_len >= len(df):
        test_len = max(1, len(df)//10)
    test = df.iloc[-test_len:].copy()
    train_val = df.iloc[:-test_len].copy()
    if len(train_val) > 100:
        cut = int(len(train_val) * 0.85)
        train, val = train_val.iloc[:cut].copy(), train_val.iloc[cut:].copy()
    else:
        train, val = train_val.copy(), pd.DataFrame(columns=train_val.columns)
    return train, val, test

def metrics_series(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    mae = float(np.mean(np.abs(y_true - y_pred)))
    rmse = float(np.sqrt(np.mean((y_true - y_pred)**2)))
    denom = np.maximum(1e-8, np.where(y_true==0, 1e-8, y_true))
    mape = float(np.mean(np.abs((y_true - y_pred) / denom)) * 100.0)
    return dict(rmse=rmse, mae=mae, mape=mape)

def save_plot_ts(ts_df: pd.DataFrame, cols, title, path: Path):
    plt.figure(figsize=(10,4))
    for c in cols:
        plt.plot(ts_df.index, ts_df[c], label=c)
    plt.title(title); plt.xlabel("Time"); plt.ylabel("Power")
    plt.legend(loc="best"); plt.tight_layout()
    plt.savefig(path, bbox_inches="tight"); plt.close("all")

def rel(p: Path, base: Path) -> str:
    try: return p.relative_to(base).as_posix()
    except Exception: return p.as_posix()

def write_text(path: Path, text: str):
    path.write_text(text, encoding="utf-8")
    print("Wrote:", rel(path, Path.cwd()))


if not S2_METRICS.exists():
    write_text(REPORTS_DIR / "SDS-CP036-powercast_Wk04_Section3_Business_Report.md",
               "# SDS-CP036-powercast – Week 4 Section 3: Model Selection & Trade-offs\n\n"
               f"Profile: `{PROFILE}`\n\n**Note:** Section 2 metrics not found: {rel(S2_METRICS, BASE_DIR)}")
    raise SystemExit(0)

dfm = pd.read_csv(S2_METRICS)
# choose champion by lowest MAPE per zone
champs = dfm.sort_values(["zone","mape"]).groupby("zone").head(1).reset_index(drop=True)
champ_csv = REPORTS_DIR / "champion_models.csv"
champs.to_csv(champ_csv, index=False)

s3_md = REPORTS_DIR / "SDS-CP036-powercast_Wk04_Section3_Business_Report.md"
write_text(s3_md, f"""
# SDS-CP036-powercast – Week 4 Section 3: Model Selection & Trade-offs

Profile: `{PROFILE}`

### Key Questions Answered
**Q: What trade-offs did you consider when selecting your final model(s) for each zone?**  
**A:** We weighed **accuracy vs. runtime vs. interpretability**. If a complex model (e.g., XGBoost) beats SARIMAX meaningfully on **MAPE**, it’s chosen; otherwise, we prefer **simpler** models that are easier to operate.

**Q: How did you compare and document performance of different models?**  
**A:** We use the **Section 2 test metrics** and select the lowest **MAPE** per zone (champion). Full details are saved to `{rel(champ_csv, BASE_DIR)}`.

**Q: What factors influenced final selection?**  
**A:**  
- **Interpretability:** SARIMAX is clearer for audits; XGB is stronger for non-linear spikes.  
- **Accuracy:** Champion is the lowest-MAPE model per zone.  
- **Compute cost:** Preference for models that meet SLAs with modest resources.

### Outcome
Champion models per zone (lowest MAPE): `{rel(champ_csv, BASE_DIR)}`

### Business Value (Executive View)
- **Clear, defendable choices** with traceable criteria.  
- **Right tool for each zone**—maximize accuracy where it matters, keep operations simple where gains are marginal.
""")

# upsert consolidated
def upsert_section(consolidated_path: Path, section_idx: int, title: str, new_block: str):
    if not consolidated_path.exists():
        consolidated_path.write_text("# SDS-CP036-powercast – Week 4 (Business Report)\n\n", encoding="utf-8")
    text = consolidated_path.read_text(encoding="utf-8")
    hdr_any = re.compile(r"(?m)^\#\s*SDS-CP036-powercast\s*[–-]\s*Week\s*4\s*Section\s*[1-5]\s*:\s*.*$")
    hdr_me = re.compile(fr"(?m)^\#\s*SDS-CP036-powercast\s*[–-]\s*Week\s*4\s*Section\s*{section_idx}\s*:\s*{title}\s*$")
    m = hdr_me.search(text)
    if m:
        start = m.start()
        m2 = hdr_any.search(text, pos=m.end())
        end = m2.start() if m2 else len(text)
        text = (text[:start] + text[end:]).strip()
    text = text.rstrip() + "\n\n" + new_block.strip() + "\n"
    consolidated_path.write_text(text, encoding="utf-8")

upsert_section(CONSOLIDATED, 3, "Model Selection \& Trade-offs", s3_md.read_text(encoding="utf-8"))
print("✅ Upserted Section 3 into consolidated:", rel(CONSOLIDATED, BASE_DIR))


Wrote: /home/6376f5a9-d12b-4255-9426-c0091ad440a7/Powercast/results/Wk04_Section3_dev/reports/SDS-CP036-powercast_Wk04_Section3_Business_Report.md
✅ Upserted Section 3 into consolidated: SDS-CP036-powercast_Wk04_Report_Business.md
